In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("train.csv")

# Check data
print(df.head())
print(df.shape)
print(df.info())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
(8

In [ ]:
X = df.drop("Survived", axis=1)
y = df["Survived"]

In [ ]:
X = X.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Numerical columns
numerical_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

# Categorical columns
categorical_features = [
    "Sex",
    "Embarked"
]


# Numerical preprocessing
numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)


# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)


# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

dt_depth_2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=2,
            random_state=42
        ))
    ]
)

# Train
dt_depth_2.fit(X_train, y_train)

# Predictions
train_pred_dt2 = dt_depth_2.predict(X_train)
test_pred_dt2 = dt_depth_2.predict(X_test)

# Accuracy
train_acc_dt2 = accuracy_score(y_train, train_pred_dt2)
test_acc_dt2 = accuracy_score(y_test, test_pred_dt2)

print("Decision Tree Depth = 2")
print("Training Accuracy:", train_acc_dt2)
print("Testing Accuracy:", test_acc_dt2)

Decision Tree Depth = 2
Training Accuracy: 0.8047752808988764
Testing Accuracy: 0.7597765363128491


In [ ]:
dt_depth_10 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(
            max_depth=10,
            random_state=42
        ))
    ]
)

# Train
dt_depth_10.fit(X_train, y_train)

# Predictions
train_pred_dt10 = dt_depth_10.predict(X_train)
test_pred_dt10 = dt_depth_10.predict(X_test)

# Accuracy
train_acc_dt10 = accuracy_score(y_train, train_pred_dt10)
test_acc_dt10 = accuracy_score(y_test, test_pred_dt10)

print("Decision Tree Depth = 10")
print("Training Accuracy:", train_acc_dt10)
print("Testing Accuracy:", test_acc_dt10)

Decision Tree Depth = 10
Training Accuracy: 0.9396067415730337
Testing Accuracy: 0.8044692737430168


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]
)

# Train
rf.fit(X_train, y_train)

# Predictions
train_pred_rf = rf.predict(X_train)
test_pred_rf = rf.predict(X_test)

# Accuracy
train_acc_rf = accuracy_score(y_train, train_pred_rf)
test_acc_rf = accuracy_score(y_test, test_pred_rf)

print("Random Forest - 100 Trees")
print("Training Accuracy:", train_acc_rf)
print("Testing Accuracy:", test_acc_rf)

Random Forest - 100 Trees
Training Accuracy: 0.9831460674157303
Testing Accuracy: 0.8156424581005587


In [ ]:
results = pd.DataFrame({
    "Model": [
        "Decision Tree (Depth=2)",
        "Decision Tree (Depth=10)",
        "Random Forest (100 Trees)"
    ],

    "Training Accuracy": [
        train_acc_dt2,
        train_acc_dt10,
        train_acc_rf
    ],

    "Testing Accuracy": [
        test_acc_dt2,
        test_acc_dt10,
        test_acc_rf
    ]
})

# Accuracy percentage
results["Training Accuracy"] = results["Training Accuracy"] * 100
results["Testing Accuracy"] = results["Testing Accuracy"] * 100

print(results)

                       Model  Training Accuracy  Testing Accuracy
0    Decision Tree (Depth=2)          80.477528         75.977654
1   Decision Tree (Depth=10)          93.960674         80.446927
2  Random Forest (100 Trees)          98.314607         81.564246
